In [3]:
# -*- encoding: utf-8 -*-
import torch.nn as nn
import torch

class SiameseNetwork(nn.Module):
    def __init__(self):
        super(SiameseNetwork, self).__init__()
        self.cnn1 = nn.Sequential(
            nn.Conv2d(1, 20, kernel_size=5),   # input_shape:(,1,28,28),output:(20,24,24)
            nn.MaxPool2d(2, stride=2),         # output:(20,12,12)
            nn.Conv2d(20, 50, kernel_size=5),  # output:(50,8,8)
            nn.MaxPool2d(2, stride=2))     # output:(50,4,4)

        self.fc1 = nn.Sequential(
            nn.Linear(50 * 4 * 4, 500),
            nn.ReLU(inplace=True),
            nn.Linear(500, 10),
            nn.Linear(10, 2))

    def forward_once(self, x):
        output = self.cnn1(x)
        output = output.view(output.size()[0], -1)
        output = self.fc1(output)
        return output

    def forward(self, input1, input2):
        output1 = self.forward_once(input1)
        output2 = self.forward_once(input2)
        return output1, output2

In [7]:
model = SiameseNetwork()
input1 = torch.randn(4,1,28,28)
input2 = torch.randn(4,1,28,28)
out = model(input1, input2)
print(f'input1.shape:{input1.shape}')
print(f'out.shape:{out[0].shape}')

input1.shape:torch.Size([4, 1, 28, 28])
out.shape:torch.Size([4, 2])


In [5]:
input1.shape

torch.Size([4, 1, 28, 28])

In [9]:
def loss(x0,x1,y, margin=1): 
    diff = x0 - x1    # diff.shape=[4,2]
    dist_sq = torch.sum(torch.pow(diff, 2), 1) # dist_sq.shape=[4]
    dist = torch.sqrt(dist_sq)
    mdist = margin - dist
    dist = torch.clamp(mdist, min=0.0)
    loss = y * dist_sq + (1 - y) * torch.pow(dist, 2)
    loss = torch.sum(loss) / 2.0 / x0.size()[0]
    return loss

loss_val = loss(out[0],out[1],1)
print(f'loss_val:{loss_val}')

loss_val:0.0010019479086622596


In [11]:
torch.max(out[0],1)

torch.return_types.max(
values=tensor([0.0584, 0.0715, 0.0595, 0.0383], grad_fn=<MaxBackward0>),
indices=tensor([1, 1, 1, 1]))